# 01 · Data quality & exploratory analysis

**AeroCast-AI · Project 21 · Phase 1.** Dataset: `nycflights13`: every 2013 departure from EWR, JFK and LGA (336,776 flights, 105 destinations, 16 carriers).

| Stage | Notebook |
|---|---|
| **1. Data quality & EDA** | **this notebook** |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| 3. Model training | `03_model_training.ipynb` |
| 4. Evaluation | `04_evaluation.ipynb` |

What this notebook establishes:
1. The data is **internally consistent**, and its missing values have a clear meaning (cancellations and diversions).
2. The **decision time** (20:00 on day D-1), and which columns are therefore allowed as features.
3. The patterns in the data that motivate each feature family and the evaluation design.

All logic lives in `src/aerocast/` (`data`, `eda`, `features`, `plots`). This notebook only calls it and explains the output.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from aerocast import config, data, eda, features, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 90)

## 1. Load

`data.load_raw()` parses the date, builds a scheduled-departure timestamp, and sorts chronologically. `data.add_target()` adds `is_cancelled`, `is_diverted`, the target `is_delayed`, and the time each outcome became known (used by the decision clock, notebook 02).

In [2]:
raw = data.load_raw()
df = data.add_target(raw)
print(raw.shape)
raw.head()

(336776, 23)


,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,origin,dest,air_time,distance,hour,minute,time_hour,name,date,sched_dep_dt
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.,2013-01-01,2013-01-01 05:15:00
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.,2013-01-01,2013-01-01 05:29:00
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.,2013-01-01,2013-01-01 05:40:00
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways,2013-01-01,2013-01-01 05:45:00
4,5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,...,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,United Air Lines Inc.,2013-01-01,2013-01-01 05:58:00


## 2. Data quality audit

### 2.1 Missingness

In [3]:
eda.missingness(raw)

,n_missing,pct
arr_delay,9430,2.8001
air_time,9430,2.8001
arr_time,8713,2.5872
dep_time,8255,2.4512
dep_delay,8255,2.4512
tailnum,2512,0.7459


The missing values are **structural, not random**:
* `dep_time` / `dep_delay` missing (8,255 rows) means the flight was **cancelled**.
* `arr_delay` / `air_time` missing while `dep_time` is present means it was **diverted** (or the arrival wasn't recorded).
* `tailnum` missing (2,512 rows) means no aircraft was assigned, and every one of these is a cancellation (§2.2).

So imputing these columns would be wrong. They encode outcomes, and outcomes are what we predict.

### 2.2 Integrity: duplicates, cancellations, diversions

In [4]:
eda.quality_audit(df).to_frame()

,value
rows,336776
duplicate id,0
"duplicate (date, carrier, flight, origin, sched_dep)",0
cancelled (no dep_time),8255
"diverted (departed, no arr_delay)",1175
missing tailnum,2512
missing tailnum AND cancelled,2512


### 2.3 Internal consistency between redundant columns

In [5]:
eda.consistency_checks(df).to_frame()

,value
hour/minute match sched_dep_time (share),1.0
"dep_delay = dep_time - sched_dep_time, mod 24h (share)",1.0
routes with >1 distinct distance,2.0
air_time <= 0,0.0


### 2.4 The two routes with more than one distance

In [6]:
eda.distance_anomalies(df)

flights      first       last
origin dest distance                               
EWR    EGE  1725           51 2013-03-01 2013-12-31
            1726           59 2013-01-01 2013-02-28
JFK    EGE  1746           44 2013-03-01 2013-12-31
            1747           59 2013-01-01 2013-02-28

Both routes go to **EGE** (Eagle County, CO), and both change by exactly **1 mile on 1 March 2013** (1726 → 1725 and 1747 → 1746). That's a dated update of the reference distance, not random noise or a data-entry error. A 1-mile difference on a 1,700-mile route has no practical effect on `log_distance`, so the values are kept as recorded.

**Conclusion:** no duplicates, `dep_delay` is consistent with the clock times for 100% of flights, and no cleaning is needed beyond handling cancelled and diverted flights.

## 3. Decision time and leakage audit

> **Predictions for all flights on day D are issued at 20:00 (NYC) on day D-1.** Allowed inputs: the published timetable for D, plus outcomes **already known at that moment**.

Every raw column is classified below. Only the first three rows may describe the flight being predicted.

In [7]:
eda.leakage_audit()

,columns,source,known at D-1?,use
0,"year, month, day, hour, minute, sched_dep_time, sched_arr_time",schedule,yes,calendar & time-of-day features
1,"carrier, flight, origin, dest, distance",schedule,yes,identity / route features
2,tailnum,aircraft assignment,yes (assumed),rotation features; assumes routing is published day-ahead
3,"dep_time, dep_delay",operations,NO - after departure,never a feature; past outcomes known by 20:00 on D-1 only
4,"arr_time, arr_delay, air_time",operations,NO - after arrival,target; past outcomes known by 20:00 on D-1 only (arrival on NYC clock)
5,"id, name, time_hour",bookkeeping,-,dropped (row id / redundant)


Two consequences that are easy to get wrong:
* `arr_delay` / `dep_delay` **are** used, but only aggregated over outcomes that were already known at the decision time. A D-1 flight that lands at 21:30 does **not** count for D (notebook 02 quantifies this).
* **Cancelled flights stay in the schedule counts.** At D-1 nobody knows a flight will be cancelled. They are dropped only from the target.
* **No weather.** Observed weather for day D is not known at 20:00 on D-1, so it would leak at this decision clock. Weather could only enter as **archived forecasts issued before that time**; this project uses none (stated as a limitation in notebook 04).

## 4. Target

`is_delayed = arr_delay ≥ 15 min`, the BTS definition of a late arrival. Cancelled and diverted flights (2.8%) have no arrival delay and are excluded from modelling. Counting them as delayed instead is tested as a **sensitivity analysis** in notebook 04.

In [8]:
fig = plots.delay_histogram(df["arr_delay"]); plots.save(fig, "eda_delay_hist"); fig

<Figure size 700x300 with 1 Axes>

In [9]:
eda.delay_shares(df).to_frame()

,share of completed flights
>= 15 min,0.2447
>= 30 min,0.1613
>= 60 min,0.0865


Strongly right-skewed with a long tail. **24.5%** of completed flights are late at the 15-minute threshold. That's moderate class imbalance, so **PR-AUC** is the main ranking metric (notebook 04, §1). The classes are not re-weighted, so that predicted probabilities stay interpretable.

## 5. When are flights late?

In [10]:
fig = plots.rate_by_time(eda.rate_by(df, "month")["rate"],
                         eda.rate_by(df, "hour")["rate"].dropna(),
                         eda.rate_by(df, df["date"].dt.dayofweek)["rate"])
plots.save(fig, "eda_rate_by_time"); fig

<Figure size 1300x330 with 3 Axes>

In [11]:
eda.rate_by(df, "month").T

month,1,2,3,4,5,6,7,8,9,10,11,12
rate,0.2349,0.2378,0.236,0.2875,0.2267,0.3152,0.3221,0.246,0.1372,0.1756,0.181,0.3355
n,26398.0000,23611.0000,27902.000,27564.0000,28128.0000,27075.0000,28293.0000,28756.000,27010.0000,28618.0000,26971.000,27020.0000


* **Time of day is the dominant pattern.** The delay rate rises from 10% at 05:00 to 37% at 21:00, because delay *propagates* through the day. → cyclic departure/arrival time features.
* **Strong seasonality, but only one year.** June–July and December peak, while September–October are the calmest months. → month features are tested by CV in notebook 02.
* **The chronological split is a built-in stress test.** The calmest months (Sep–Oct) become the calibration window and the worst month (Dec) is in test.

In [12]:
fig = plots.rate_by_group(eda.rate_by(df, "carrier"), eda.rate_by(df, "origin"))
plots.save(fig, "eda_rate_by_carrier_origin"); fig

<Figure size 1100x360 with 2 Axes>

Carrier differences are large (about 3×), but several carriers are tiny (OO has 32 flights). → one-hot encoding with `min_frequency=50`, and history rates shrunk toward a prior (notebook 02).

## 6. Is yesterday informative?

In [13]:
daily = eda.daily_rate(df)
acf = eda.autocorrelation(daily)
fig = plots.daily_rate_and_acf(daily, acf); plots.save(fig, "eda_daily_autocorr"); fig

<Figure size 1200x320 with 2 Axes>

In [14]:
acf.round(3).to_frame().T

,1,2,3,4,5,6,7,8,9,10,11,12,13,14
autocorr,0.398,0.128,0.161,0.152,0.107,0.139,0.112,0.083,0.089,0.103,0.18,0.148,0.145,0.195


Lag-1 autocorrelation is about **0.40**, then it drops quickly to about 0.1–0.15. Disruption is bursty with short memory. That motivates **day-level** history features (the last 24 hours known before the decision) alongside **entity-level** ones (route, carrier, aircraft, flight number over 7/14 days).

## 7. Congestion vs delay, and the time-of-day confound

In [15]:
curve = eda.congestion_curve(df)
fig = plots.congestion_curve(curve); plots.save(fig, "eda_congestion"); fig

<Figure size 700x300 with 1 Axes>

Busier departure banks have more delays. But busy banks cluster in the evening, when delays are high anyway, so the curve above mixes the two effects. To separate them, we split flights into bank-size **terciles within each departure hour**, so the hour is held fixed:

In [16]:
within = eda.congestion_within_hour(df)
fig = plots.congestion_within_hour(within); plots.save(fig, "eda_congestion_within_hour"); fig

<Figure size 700x320 with 1 Axes>

In [17]:
display(within.round(3).T)
eda.within_hour_gap(within).to_frame("value")

hour,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
tercile,,,,,,,,,,,,,,,,
low,0.121,0.094,0.157,0.145,0.159,0.174,0.174,0.198,0.257,0.260,0.314,0.270,0.306,0.357,0.296,0.284
mid,0.112,0.122,0.189,0.189,0.199,0.160,0.212,0.243,0.266,0.312,0.340,0.355,0.375,0.346,0.356,0.357
high,0.123,0.142,0.152,0.172,0.192,0.226,0.236,0.270,0.274,0.331,0.266,0.379,0.317,0.346,0.402,0.465


,value
mean high-low gap,0.0454
hours with high > low,13.0000
hours,16.0000


**Congestion still matters with the hour held fixed.** The busiest third of banks is about **4.5 percentage points** later than the quietest third, and is later in 13 of 16 hours. So part of the raw congestion effect is time of day, but not all of it. Whether congestion features add value *on top of* time of day and history features is a separate question, answered by the ablation in notebook 04.

## 8. Multicollinearity of the history features (training window only)

In [18]:
X, groups = features.build_features(df)
train = (df["date"] > config.WARMUP_END) & (df["date"] <= config.SPLITS["train"][1]) & df["is_delayed"].notna()
hist_rates = [c for c in groups["history_entity"] + groups["history_day"] if "rate" in c or "mean" in c]
corr = X.loc[train, hist_rates].corr()
fig = plots.corr_heatmap(corr, "History features: correlation (train window)"); plots.save(fig, "eda_history_corr_heatmap"); fig

<Figure size 796x696 with 2 Axes>

In [19]:
pairs = corr.where(~pd.DataFrame(__import__('numpy').tri(len(corr), dtype=bool), index=corr.index, columns=corr.columns)).stack()
pairs.sort_values(key=abs, ascending=False).head(8).rename("r").to_frame()

r
hist_rate_origin_1d       hist_meandelay_origin_1d           0.9575
hist_rate_network_1d      hist_rate_origin_1d                0.9154
                          hist_meandelay_origin_1d           0.9056
hist_rate_origin_7d       hist_rate_network_7d               0.9011
hist_rate_origin_dest_14d hist_rate_carrier_origin_dest_14d  0.8707
hist_rate_dest_14d        hist_rate_origin_dest_14d          0.8428
hist_rate_carrier_7d      hist_rate_carrier_14d              0.8404
hist_rate_origin_7d       hist_rate_carrier_7d               0.7390

The history rates are strongly inter-correlated: nested windows on the same key (e.g. carrier 7 d vs 14 d), and the network and origin rates that move together from day to day. Consequences for modelling:
* **Individual coefficients become unstable.** Correlated features can share credit or take opposite signs. LogReg is therefore **L2-regularised**, with C chosen by CV (notebook 03).
* **Single-feature importance is misleading**, because a correlated partner stands in when one feature is shuffled. Notebook 04 uses **grouped** permutation importance, which shuffles a whole family at once.

## 9. EDA → design decisions

In [20]:
eda.design_decisions()

,EDA observation,design decision
0,Missingness = cancellations / diversions,"drop from target, keep in schedule (congestion) counts"
1,Strong time-of-day gradient (10% -> 37%),cyclic dep/arr time + raw hour
2,"One year; seasonal months; Sep-Oct calm, Dec extreme",no month feature; chronological split; calibration studied under shift
3,Short-memory daily shocks (lag-1 autocorr ~0.4),1-day / 7-day lagged rates (day-level group)
4,Stable carrier / route / aircraft propensities; some tiny groups,"7/14-day entity rates with Bayesian shrinkage (CV-tuned m, window); one-hot min_freque..."
5,Congestion correlates with delay but is confounded with hour,"scheduled-bank counts, kept and tested by ablation"
6,Leakage risk from post-departure columns; 31% of D-1 arrivals land after 20:00,decision clock 20:00 on D-1; outcomes enter history only once known; two leakage tests
7,"History rates strongly inter-correlated (same keys, nested windows)",L2-regularised LogReg; grouped (not per-feature) permutation importance
